# One check, taken apart

The reconciliation loop closes the difference between what should exist and what
does. It does that one **check** at a time, and a check is three steps:

| | |
|---|---|
| **observe** | look at storage, make `current_state` say what is really there |
| **gap** | what should exist, minus what does |
| **act** | submit a job, record something, or do nothing |

This notebook runs those steps by hand on a single reach, with a real
`build_model` container, so you can see what each one decides before the next
one happens.

The thing to watch for: **a check never waits for its job.** It submits, writes
down that it did, and ends. Seeing one job through therefore takes more than one
check — which is the whole point, not an inconvenience.

**Before running:** `docker compose up -d db minio minio-init`, and the
`build_model` image available locally.

In [ ]:
import time

import pandas as pd

from recon import activity, check, db, gap, jobs, observe, processing, queue, storage
from recon.config import settings
from recon.workers import LocalDockerRunner

pd.set_option("display.max_colwidth", 40)

print(f"database  {settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}")
print(f"storage   s3://{settings.artifacts_s3_bucket} via {settings.aws_endpoint_url}")
print(f"job image {settings.build_model_image}")

## Where we are

Six tables. Three describe the world — intent, what exists, and the run ledger —
and two are the reconciler's own notes. Nothing here is a queue: the queue is a
question asked of these tables.

In [ ]:
pd.DataFrame(db.table_counts())

## 1. Author some intent

`desired_state` is the input to the system. A row saying a reach should exist is
all it takes for the loop to start caring about it.

We load a real network from a GeoPackage so the geometry is realistic —
`build_model` reads it from the database itself.

In [ ]:
import sys
sys.path.insert(0, "../scripts")
from seed import load_network  # parsing only; we insert through recon.db

network = load_network("../testdata/network.gpkg")

with db.connect() as conn:
    conn.execute("TRUNCATE reach_network CASCADE")   # start clean
    for r in network:
        # The GeoPackage says which reaches are terminal but not why. Only
        # modify_network can tell an outlet from a lake or coast break, so
        # anything terminal is recorded as an outlet here.
        reason = r["terminal_reason"] or ("outlet" if r["is_terminal"] else None)
        conn.execute(
            """INSERT INTO reach_network (reach_id, reach_to_id, is_terminal, is_headwater,
                   terminal_reason, total_da_sqkm, stream_order, slope, geom)
               VALUES (%s,%s,%s,%s,%s,%s,%s,%s, ST_GeomFromText(%s, 5070))""",
            (r["reach_id"], r["reach_to_id"], r["is_terminal"], r["is_headwater"],
             reason, r["total_da_sqkm"], r["stream_order"], r["slope"], r["geom"]))
    conn.execute("INSERT INTO desired_state (reach_id, q_lower_bound, q_upper_bound) "
                 "SELECT reach_id, 0, 500 FROM reach_network")

pd.DataFrame(db.table_counts())

## 2. The queue is a question, not a list

Nothing was pushed anywhere. Asking the database which reaches need looking at
*is* the queue, which is why a reconciler that dies mid-sweep loses nothing —
the next one asks again.

Each row says why it is here.

In [ ]:
due = pd.DataFrame(queue.due_reaches())
print(f"{len(due)} reaches due a check")
due.head(8)

## 3. A check, one step at a time

Rather than call `run_check` straight away, here are its three steps
individually on one reach, so you can see each decision.

### observe — what does storage actually hold?

Note that this is also how a *deletion* is noticed. Recording a finished job and
noticing a removed file are the same code running in two directions, which is
why there is exactly one definition of what it means for a model to exist.

In [ ]:
reach_id = int(due.iloc[0]["reach_id"])
print(f"working on reach {reach_id}\n")

# Storage outlives the database, so this reach may already have a model from an
# earlier run — in which case there would be no gap and nothing to watch. Delete
# it, which is also the supported way to undo a model.
s3 = storage.get_s3_client()
bucket, prefix = storage.parse_s3_path(storage.model_base_path(reach_id))
for obj in s3.list_objects_v2(Bucket=bucket, Prefix=prefix).get("Contents", []):
    s3.delete_object(Bucket=bucket, Key=obj["Key"])

seen = observe.observe_reach(reach_id)
print("observe:", seen)

### gap — what should exist, minus what does

`gap.calculate` is a pure function: no database, no storage, no clock. It takes
a snapshot of facts and returns a decision. That is what makes it possible to
say the loop gives the same answer every time for the same inputs, and to test
it without any infrastructure at all.

In [ ]:
snapshot = check.load_snapshot(reach_id)
print("snapshot:", snapshot)
print("decision:", gap.calculate(snapshot))

### act — submit, and stop

`run_check` does all three and acts on the decision. Watch what it returns:
it submits a container and comes straight back. It does **not** wait for the
build.

In [ ]:
import os

# The job runs as a sibling container on the compose network, so it needs the
# service names (minio, db), not localhost. Volumes carry the local LULC
# override when USGS requester-pays access is not available.
env_vars = {"AWS_ENDPOINT_URL": "http://minio:9000"}
for key in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN", "AWS_REQUEST_PAYER"):
    if os.environ.get(key):
        env_vars[key] = os.environ[key]

volumes = [f"{settings.docker_data_dir}:/data:ro"] if settings.docker_data_dir else []

runner = LocalDockerRunner(
    image=settings.build_model_image,
    network=settings.docker_network,
    env_vars=env_vars,
    platform=settings.docker_platform,
    volumes=volumes,
)
print("runner:", runner.image, "on", runner.network)
print("volumes:", volumes)

result = check.run_check(reach_id, runner)
print(result)

## 4. The check ended; the job did not

The job is running, and the fact that it is running lives in the **database**,
not in this notebook's memory. Kill the kernel now and nothing is lost.

Check the same reach again and it will not resubmit — the in-flight marker is
what stops that, and it is a rule inside the gap calculation rather than an
exclusion in the queue.

In [ ]:
flying = processing.in_flight()
if flying:
    print(pd.DataFrame(flying)[["reach_id", "current_step", "current_step_ref", "elapsed"]])
else:
    print("nothing in flight - the previous cell found no gap, so it submitted nothing")
print()
print("checking again:", check.run_check(reach_id, runner))

## 5. Hearing back

A second sweep asks the execution system what became of the jobs in flight. Its
work list is also just a query, so it too survives a restart.

It records nothing about what exists — it only clears the marker and asks for a
check. Storage is what decides whether anything was produced.

In [ ]:
deadline = time.time() + 900
while time.time() < deadline:
    outcomes = jobs.status_pass(runner)
    if not outcomes:
        print("nothing in flight")
        break
    print(f"{time.strftime('%H:%M:%S')}  {outcomes[0]['status']:<10} {outcomes[0]['action']}")
    if outcomes[0]["status"] in ("succeeded", "failed"):
        break
    time.sleep(20)

## 6. Only now does anything get recorded

The job finished, but nothing has been written about what exists yet — the
status pass deliberately did not do that. It takes another check, which observes
storage and finds the manifest there.

`build_model` writes `model_manifest.json` last, so its presence is the signal
that a model is complete rather than half-written.

In [ ]:
print(check.run_check(reach_id, runner))
print()
print(db.one("SELECT model_id, build_model_version, confirmed_at FROM current_state WHERE reach_id = %s", (reach_id,)))

## 7. Satisfied

`reach_status` derives the state — the table stores only `halted`, everything
else is read off the columns that already say it, so the database cannot hold
two opinions about one reach.

In [ ]:
pd.DataFrame(db.query(
    "SELECT reach_id, state, model_id, desired_revision, applied_revision, has_gap "
    "FROM reach_status WHERE reach_id = %s", (reach_id,)))

## 8. What happened, in order

`reach_activity` is the only table with a time dimension. Nothing reads it to
make a decision — a check works everything out from current state — which is
exactly why it is safe for it to be trimmed or lost.

In [ ]:
pd.DataFrame(activity.recent(10, reach_id=reach_id))[["action", "outcome", "revision", "detail", "took"]]

## What this showed

- A check is short and never waits. Three checks saw one job through: one
  submitted, one found it still running and did nothing, one recorded the result.
- Nothing was believed because a job said so. `current_state` was written only
  after the manifest was seen in storage.
- Every fact that had to survive a crash was in the database before the step
  that wrote it returned.

`02_cases.ipynb` takes the same machinery and shows what it does when things go
wrong — deletions, failures, lost jobs, duplicate submissions.